# Rotated surface code — **subset** joint lattice-surgery measurement (obstacle-aware routing)

**[DEMO]** — all logic lives in the package; this notebook only imports and demonstrates.  Routing /
construction: [`subset_routing.py`](../../lightstim/qec_code/surface_code/rotated/subset_routing.py);
the report front-end [`subset_report.py`](../../lightstim/qec_code/surface_code/rotated/subset_report.py)
exposes the single public entry point `report_subset_joint` used below.

Measure the joint `M(∏ᵢ P̄ᵢ)` of only a **subset** of many placed patches; the non-target patches are
**obstacles** the routed `d`-wide ancilla bus avoids.

**Input** — just the native `PatchSpec(name, origin, distance, measured_logical, orientation)` **array**
(the same class the N-patch API uses) + a `target = [(name, "X"|"Z"), …]` list saying which patches are
measured (the rest are **obstacles**).  `origin_of(a,b,d)` places a patch on coarse cell `(a,b)`.
**No hand-written corridor:** the underlying `route_and_build(patches, target)` finds *and* verifies the
bus automatically — propose-and-verify with retry, trying the **standard** construction first and the
**convex-corner cut** as a fallback (see the Summary).

**Every example PASSES** the strict **12-point acceptance** gate; bent geometries fall back automatically to
the **convex-corner cut** (removing the bend's outer qubit so the convex 90° corner becomes a genuine
**weight-3** stabilizer — fully CSS, no twist).  Each example calls `report_subset_joint`, which shows
**four** things (each behind a `show_*` toggle, all on by default): **(1)** the auto-routed path, **(2)** the
12-point acceptance checklist, **(3)** the data-qubit layout, **(4)** the Stim `detslice-with-ops` diagram —
and returns the verified `SubsetRoute` for programmatic use.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

# PatchSpec is lightstim's NATIVE spec class (defined in .bent_layout, exported at the rotated package
# level) -- the very same input the N-patch generator API (build_rotated_multi_patch_joint_layout)
# uses.  NOT a wrapper.  report_subset_joint (subset_report.py) is the single public entry point:
# route + build + verify via route_and_build, then the four report artefacts (each behind a show_*
# toggle, all default True).  origin_of is subset_routing's public coarse-grid placement helper.
from lightstim.qec_code.surface_code.rotated import PatchSpec, report_subset_joint
from lightstim.qec_code.surface_code.rotated.subset_routing import origin_of

D = 3   # code distance for the demos

## Example 1 — subset joint `M(X̄₁ Z̄₂ Z̄₃)`: 9 patches, measure only 3

Nine `PatchSpec`s; `target` selects `X1` + `Z2` + `Z3`; the six `B*` are **obstacles** the bus threads
between (`keepout = 1`).  `report_subset_joint` finds and verifies the corridor automatically — no `route` argument.

In [ ]:
patches = [PatchSpec("X1", origin_of(0,  0, D), D, "X", "X_horizontal"),
           PatchSpec("Z2", origin_of(2,  1, D), D, "Z", "X_horizontal"),
           PatchSpec("Z3", origin_of(4, -1, D), D, "Z", "X_horizontal"),
           PatchSpec("B1", origin_of(0,  3, D), D, "X", "X_horizontal"),
           PatchSpec("B2", origin_of(2,  3, D), D, "X", "X_horizontal"),
           PatchSpec("B3", origin_of(4,  3, D), D, "X", "X_horizontal"),
           PatchSpec("B4", origin_of(0, -3, D), D, "X", "X_horizontal"),
           PatchSpec("B5", origin_of(2, -3, D), D, "X", "X_horizontal"),
           PatchSpec("B6", origin_of(4, -3, D), D, "X", "X_horizontal")]
target  = [("X1", "X"), ("Z2", "Z"), ("Z3", "Z")]
r = report_subset_joint(patches, target, "Subset joint  M(X̄1 Z̄2 Z̄3)")   # corridor found automatically

## Example 2 — subset joint `M(X̄₁ Z̄₃)`: drop `Z2` (obstacle between targets)

Same nine patches, now `target` selects only `X1` and `Z3`; `Z2` becomes an obstacle **between** them.
`route_and_build` auto-finds a clean below-attach corridor (`X1` exits its bottom face and runs straight
across to `Z3`) and verifies it with the **standard** construction — a genuine *non-adjacent* subset.

In [ ]:
patches = [PatchSpec("X1", origin_of(0,  0, D), D, "X", "X_horizontal"),
           PatchSpec("Z2", origin_of(2,  1, D), D, "Z", "X_horizontal"),
           PatchSpec("Z3", origin_of(4, -1, D), D, "Z", "X_horizontal"),
           PatchSpec("B1", origin_of(0,  3, D), D, "X", "X_horizontal"),
           PatchSpec("B2", origin_of(2,  3, D), D, "X", "X_horizontal"),
           PatchSpec("B3", origin_of(4,  3, D), D, "X", "X_horizontal"),
           PatchSpec("B4", origin_of(0, -3, D), D, "X", "X_horizontal"),
           PatchSpec("B5", origin_of(2, -3, D), D, "X", "X_horizontal"),
           PatchSpec("B6", origin_of(4, -3, D), D, "X", "X_horizontal")]
target  = [("X1", "X"), ("Z3", "Z")]                      # drop Z2 -> it is now an obstacle
r = report_subset_joint(patches, target, "Subset joint  M(X̄1 Z̄3)  (drop Z2)")   # corridor found automatically

## Example 3 — add `B2` as a third target: `M(X̄₁ X̄_B2 Z̄₃)`

Building on Example 2's chip, with two small changes so `B2` becomes measurable: **move `Z2` one cell
right** `(2,1) → (3,1)`, and **remove `B1` and `B3`** (they flanked `B2`).  Now `B2(2,3)` is a **third
measured target** contributing `X̄`, giving `M(X̄₁ X̄_B2 Z̄₃)` (`N = 3`, so `dof = N−1 = 2`): `X1` and `B2`
fuse on the X-bus, `Z3` attaches through a mixed wall, and `Z2`/`B4`/`B5`/`B6` are obstacles.  Reaching
`B2` up high **forces a bend**, so no standard candidate verifies — `route_and_build` falls through to the
**convex-corner cut** automatically (removes the bend's outer qubit → weight-3 corner, fully CSS, no twist).

In [ ]:
patches = [PatchSpec("X1", origin_of(0,  0, D), D, "X", "X_horizontal"),
           PatchSpec("Z2", origin_of(3,  1, D), D, "Z", "X_horizontal"),   # moved right (was (2,1)); obstacle
           PatchSpec("Z3", origin_of(4, -1, D), D, "Z", "X_horizontal"),
           PatchSpec("B2", origin_of(2,  3, D), D, "X", "X_horizontal"),   # now a target (was an obstacle)
           PatchSpec("B4", origin_of(0, -3, D), D, "X", "X_horizontal"),   # obstacle
           PatchSpec("B5", origin_of(2, -3, D), D, "X", "X_horizontal"),   # obstacle
           PatchSpec("B6", origin_of(4, -3, D), D, "X", "X_horizontal")]   # obstacle
target  = [("X1", "X"), ("B2", "X"), ("Z3", "Z")]          # measure M(X̄1 X̄_B2 Z̄3)
r = report_subset_joint(patches, target, "Subset joint  M(X̄1 X̄_B2 Z̄3)")   # bent corridor + corner-cut, all automatic

## Summary

* **Input** is just a `PatchSpec` array + a `target` list (which patches are measured; the rest are
  obstacles) — the same native form as the N-patch API.  **No hand-written corridor:** `route_and_build`
  finds and verifies it automatically; `report_subset_joint` wraps it into the single public entry point
  used here (returning the verified `SubsetRoute`).
* **`route_and_build` = propose-and-verify with retry** (the algorithm this notebook is built on):
  enumerate obstacle-aware candidate corridors (any anchor face, shortest-first) and verify each against
  the GF(2) oracle — **standard construction first, then the convex-corner cut** as a fallback — returning
  the first valid one.  So a clean straight/L route always wins when one exists (Examples 1–2), and the
  corner cut is used only when the geometry genuinely forces a bend (Example 3).
* **Every example passes** the strict 12-point acceptance oracle and is collision-clean where obstacles are
  present.  Each shows four things: the auto-routed path, the acceptance checklist, the data-qubit layout,
  and the Stim detslice diagram.
* **The convex-corner cut** (remove the bend's outer bus qubit so the convex 90° corner becomes a weight-3
  stabilizer — fully CSS, no twist) is what lets a bent joint verify; `route_and_build` applies it
  automatically whenever no standard candidate leaves `dof == N−1`.